In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch.nn.functional as F

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print("Ready!")
print(f"Layers: {model.config.n_layer}")
print(f"Heads:  {model.config.n_head}")

c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6059.71it/s]


Ready!
Layers: 12
Heads:  12


In [2]:
text = "The cat sat on the"
inputs = tokenizer(text, return_tensors="pt")

def generate_with_silenced_head(layer_idx, head_idx, n_tokens=10):
    head_dim = model.config.n_embd // model.config.n_head  # 64
    
    def silence_hook(module, input, output):
        start = head_idx * head_dim
        end   = start + head_dim
        modified = output[0].clone()
        modified[:, :, start:end] = 0
        return (modified,) + output[1:]
    
    hook = model.transformer.h[layer_idx].attn.register_forward_hook(
        silence_hook)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=n_tokens,
            do_sample=False
        )
    
    hook.remove()
    return tokenizer.decode(output[0])

# Normal generation
with torch.no_grad():
    normal_out = model.generate(
        **inputs, max_new_tokens=10, do_sample=False)
normal_text = tokenizer.decode(normal_out[0])
print(f"Normal: {normal_text}\n")

test_heads = [
    (0, 0), (0, 5), 
    (5, 6), (6, 7),
    (11, 7), (11, 11)
]

print(f"{'Layer':<8} {'Head':<8} Generated Text")
print("-"*60)
for layer, head in test_heads:
    generated = generate_with_silenced_head(layer, head)
    changed   = "⚠️" if generated != normal_text else "✅"
    print(f"L{layer:<7} H{head:<7} {changed} {generated}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Normal: The cat sat on the floor, and the cat was still asleep.


Layer    Head     Generated Text
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


L0       H0       ✅ The cat sat on the floor, and the cat was still asleep.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

L5       H6       ✅ The cat sat on the floor, and the cat was still asleep.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


L6       H7       ✅ The cat sat on the floor, and the cat was still asleep.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


L11      H7       ✅ The cat sat on the floor, and the cat was still asleep.

L11      H11      ✅ The cat sat on the floor, and the cat was still asleep.



In [4]:
print(f"Normal (sampling): ", end="")
torch.manual_seed(42)
with torch.no_grad():
    normal_sample = model.generate(
        **inputs, max_new_tokens=10, 
        do_sample=True, temperature=1.0
    )
print(tokenizer.decode(normal_sample[0]))

print("\nSilencing each Layer 11 head:")
print("-"*60)

for head_idx in range(12):
    head_dim = model.config.n_embd // model.config.n_head

    def silence_hook(module, input, output):
        start    = head_idx * head_dim
        end      = start + head_dim
        modified = output[0].clone()
        modified[:, :, start:end] = 0
        return (modified,) + output[1:]

    hook = model.transformer.h[11].attn.register_forward_hook(
        silence_hook)

    torch.manual_seed(42)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=10,
            do_sample=True, temperature=1.0
        )
    hook.remove()
    
    generated = tokenizer.decode(out[0])
    print(f"H{head_idx:<3}: {generated}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Normal (sampling): 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The cat sat on the shelf and was reading the next day.



Silencing each Layer 11 head:
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

H8  : The cat sat on the shelf and was reading, when in actuality he


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


H9  : The cat sat on the shelf and was reading the next day.




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


H10 : The cat sat on the shelf and was reading, when in actuality he
H11 : The cat sat on the shelf and was a little confused by the look he


* Head silencing affects text generation
* Differences are more visible with sampling
* In Layer 11, there are 3 groups of heads
* Some heads are redundant (produce the same output)
